# R01 · Cookbook Reference Lab: Basics: install, first database, records, cells, extensions

**NHyDB 0.27.1 notebook series · v1 · reference part**

These examples are based on the official
[NHyDB Python Cookbook](https://github.com/nhydb/NHyDB/blob/main/docs/cookbook/PYTHON_COOKBOOK.md) (sections 1–8), which serves as their authoritative reference.
Code and expected output are taken unchanged from the cookbook, and the same examples are run by the
NHyDB test suite.

* Every code cell is an **independent example** – run any cell on its own, or *Run All*.
* Each example creates its own temporary database folder.
* Under each example you find the output printed in the cookbook, to compare with yours.
* The main series (`../main/`) explains the ideas behind these APIs step by step.

**Contents** (15 runnable examples)

* 1. Installation, import and version
* 2. Concepts in five minutes
* 3. Your first database
* 4. Opening databases and options
* 5. Commit, sync, close and crash behavior
* 6. Records (CLASSIC layout)
* 7. Single cells, lanes, null and delete
* 8. Extensions: additional fields per record

In [1]:
import nhydb

print('nhydb', nhydb.ENGINE_VERSION, '· file format', nhydb.FILE_FORMAT_VERSION)
if nhydb.ENGINE_VERSION != '0.27.1':
    print('⚠ These examples were written for NHyDB 0.27.1; results may differ.')

nhydb 0.27.1 · file format 0.16


## 1. Installation, import and version

```bash
pip install nhydb            # prebuilt wheels: CPython 3.9–3.14 on Linux, macOS, Windows
pip install numpy            # optional, for the NumPy batch API (section 16)
```

In [2]:
import nhydb

print(nhydb.ENGINE_VERSION)        # product version of the installed package
print(nhydb.FILE_FORMAT_VERSION)   # identifier of the on-disk format
assert nhydb.ENGINE_VERSION == "0.27.1"
assert nhydb.FILE_FORMAT_VERSION == "0.16"

0.27.1
0.16


*Expected output (from the cookbook):*

```text
0.27.1
0.16
```

`FILE_FORMAT_VERSION` only changes when the file format changes; it is not a
product version.

## 2. Concepts in five minutes

* **Cell.** Every value lives at a cell `(R, u, v, lane)`: realm `R`,
  record coordinate `u`, field/position `v` and lane `NL` (0) or `NR` (1).
  All coordinates are natural numbers ≥ 1 and may be arbitrarily large.
* **N.** Each cell has a unique natural number `N`, computed from the
  address (section 9). `N` is never stored; it is a mathematical name for
  the cell that you can use to read it.
* **Realm.** A numbered address range (`create_realm`). A realm groups
  shapes, layouts, extensions and indexes.
* **Shape.** The BASE fields of a record (`v = 1 … k`) with names and types
  (`create_shape`).
* **Layout.** How logical records map onto cells. **CLASSIC** (record =
  `u`, field = `v`) is the common case; COLUMN, PACKED, CUSTOM and MATH
  are covered in sections 17 and 18.
* **Dense and sparse.** Records written in batches are stored *dense*;
  single cells, extensions and overrides are stored *sparse*. Reading
  combines both; the newest write wins.
* **Commit boundary.** The file is append-only. Data becomes durable with
  `commit()`. Everything after the last commit is discarded when the
  database is opened again.
* **Files.** A database consists of `X.nhydb` (data),
  `X.nhydb.layout.enc` (encrypted schema), `X.nhydb.layout.key` (key – keep
  it safe!) and `X.nhydb.lock` (lock file). See section 21.

## 3. Your first database

Create a database, define a realm, a shape and a CLASSIC layout, write two
records, close, reopen and read.

In [3]:
import os
import tempfile

import nhydb

folder = tempfile.mkdtemp()
path = os.path.join(folder, "shop.nhydb")

db = nhydb.create(path)
db.create_realm(1, "shop")
db.create_shape(1, 1, "order", [
    {"v": 1, "name": "id", "type": "varuint"},
    {"v": 2, "name": "customer", "type": "utf8"},
    {"v": 3, "name": "amount", "type": "f64"},
])
db.create_classic_layout(1, 1, "orders", shape_id=1)   # layout 1 in realm 1 uses shape 1

db.put_record(1, 1, {"id": 1, "customer": "Alice", "amount": 19.90})
db.put_record(1, 2, {"id": 2, "customer": "Bob", "amount": 5.50})
db.commit()                                            # make it durable
db.close()

db = nhydb.open(path)
print(db.get_record(1, 1))
print(db.get_value(1, 2, 2))                           # realm 1, u=2, v=2 -> "customer"
db.close()

{'_realm': 1, '_u': 1, '_lane': 0, '_lane_name': 'NL', '_shape': 1, 'id': 1, 'customer': 'Alice', 'amount': 19.9}
Bob


*Expected output (from the cookbook):*

```text
{'_realm': 1, '_u': 1, '_lane': 0, '_lane_name': 'NL', '_shape': 1, 'id': 1, 'customer': 'Alice', 'amount': 19.9}
Bob
```

Fields starting with `_` describe where the record lives. The same program
with a context manager (`with` commits and closes automatically):

In [4]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "notes.nhydb")

with nhydb.create(path) as db:
    db.create_realm(1, "notes")
    db.create_shape(1, 1, "note", [{"v": 1, "name": "text", "type": "utf8"}])
    db.create_classic_layout(1, 1, "notes", shape_id=1)
    db.put_record(1, 1, {"text": "buy milk"})

with nhydb.open(path) as db:
    assert db.get_record(1, 1)["text"] == "buy milk"
    print(db.closed)
print(db.closed)

False
True


*Expected output (from the cookbook):*

```text
False
True
```

> **Note:** `__exit__` calls `close(commit=True)` even if an exception
> occurred inside the `with` block. Use explicit `close(commit=False)` if
> you want to discard changes after an error (section 5).

## 4. Opening databases and options

`nhydb.open(path, **options)` is the same as `nhydb.NHyDB(path, **options)`;
`nhydb.create(path, **options)` is the same as `nhydb.NHyDB.create(...)`.
All options are keyword arguments.

| Option | `create` | `open` | Meaning |
|---|:-:|:-:|---|
| `overwrite=False` | ✓ | | replace an existing database |
| `durability="fsync"` | ✓ | ✓ | `"fsync"` or `"full"` (section 5) |
| `lock=True` | ✓ | ✓ | exclusive lock file `<db>.lock` (section 22) |
| `lock_timeout_ms=0` | ✓ | ✓ | wait for the lock instead of failing immediately |
| `layout_path`, `key_path` | ✓ | ✓ | other locations of the schema and key files (section 21) |
| `config` | ✓ | ✓ | path of a `nhydb_config.toml` (section 19) |
| `strict=True` | | ✓ | accepted, but has no effect in 0.27.1 (opening is always fail-closed) |

In [5]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "app.nhydb")
nhydb.create(path, durability="full").close()

try:
    nhydb.create(path)                       # already exists
except FileExistsError as e:
    print("FileExistsError")

db = nhydb.create(path, overwrite=True)      # start from scratch
print(str(db.path).endswith("app.nhydb"), db.durability)   # db.path is a pathlib.Path
db.close()

try:
    nhydb.open(os.path.join(os.path.dirname(path), "missing.nhydb"))
except FileNotFoundError:
    print("FileNotFoundError")

FileExistsError
True fsync
FileNotFoundError


*Expected output (from the cookbook):*

```text
FileExistsError
True fsync
FileNotFoundError
```

Useful attributes of an open database:

In [6]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "info.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "main")
    db.commit()
    print(os.path.basename(db.path), os.path.basename(db.layout_path), os.path.basename(db.key_path))
    print(db.durability, db.max_db_size_bytes)
    print(db.committed_end > 0, db.recovered_tail_bytes, db.config_source)
    print(db.realms)

info.nhydb info.nhydb.layout.enc info.nhydb.layout.key
fsync 53687091200
True 0 None
{1: {'realm_id': 1, 'name': 'main', 'description': '', 'default_shape_id': None, 'base_reserve_to': 0}}


*Expected output (from the cookbook):*

```text
info.nhydb info.nhydb.layout.enc info.nhydb.layout.key
fsync 53687091200
True 0 None
{1: {'realm_id': 1, 'name': 'main', 'description': '', 'default_shape_id': None, 'base_reserve_to': 0}}
```

`db.durability` can also be changed while the database is open
(`db.durability = "full"`).

## 5. Commit, sync, close and crash behavior

* `commit()` writes all pending data, syncs the file and moves the commit
  boundary. It returns the new boundary (a byte offset).
* Every write method has `commit=False` by default; `commit=True` commits
  right after the write.
* `sync(hard=False)` writes buffers to the file (`hard=True` also syncs the
  file) but does **not** commit.
* `close(commit=True)` commits and releases the file and the lock.
  `close(commit=False)` closes without committing.
* After `close()` the file handle and the lock are released immediately.
  Writes and uncached reads raise `NHyDBError("database is closed")`;
  calling `close()` again does nothing. Do not use a closed object for
  reading either: in 0.27.1 a record that was read before `close()` may
  still be returned from the in-memory cache.
* Uncommitted data is discarded when the database is opened again;
  `recovered_tail_bytes` tells you how many bytes were cut off.

In [7]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "crash.nhydb")
db = nhydb.create(path)
db.create_realm(1, "r")
db.create_shape(1, 1, "s", [{"v": 1, "name": "n", "type": "varuint"}])
db.create_classic_layout(1, 1, "c", shape_id=1)
db.put_record(1, 1, {"n": 100}, commit=True)   # durable
db.put_record(1, 2, {"n": 200})                # not committed
db.close(commit=False)                         # like a crash before commit

db = nhydb.open(path)
print(db.recovered_tail_bytes > 0)             # the uncommitted tail was cut off
print(db.get_record(1, 1)["n"], db.get_record(1, 2))
db.close()
db.close()                                     # no effect

try:
    db.put_record(1, 3, {"n": 300})
except nhydb.NHyDBError as e:
    print(e)

True
100 None
database is closed


*Expected output (from the cookbook):*

```text
True
100 None
database is closed
```

**Durability modes.** `"fsync"` (default) uses the operating system's file
sync. `"full"` additionally flushes the drive cache on macOS
(`F_FULLFSYNC`); on Linux and Windows both modes perform the same call.
`"full"` is slower; on macOS it raises an error if the file system does not
support it.

**There is no rollback API.** Atomicity comes from the commit boundary: to
discard work, close with `commit=False` (or let the process end) before
committing. Note that a single call that writes several blocks (for example
`put_points` across many `u` windows) is not atomic as a whole – only
`put_math_batch` rolls back completely on errors.

## 6. Records (CLASSIC layout)

In a CLASSIC layout a record is the cell row `(R, u, v = 1 … k)`: the
record id is `u`, the fields are the shape fields. There are several ways
to write records:

| Method | Writes | Addressed by |
|---|---|---|
| `put_record(layout_id, object_id, values_dict)` | one record | layout + object id |
| `put(realm_id, u, values_list)` | one record | realm + `u` |
| `put_dense_batch(realm_id, u_start, rows)` | consecutive records `u_start, u_start+1, …` | realm + `u` |
| `put_object` / `put_objects_batch` | one / many records of **any** layout (section 17) | layout + object id |

In [8]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "orders.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "shop")
    db.create_shape(1, 1, "order", [
        {"v": 1, "name": "id", "type": "varuint"},
        {"v": 2, "name": "customer", "type": "utf8"},
        {"v": 3, "name": "amount", "type": "f64"},
    ])
    db.create_classic_layout(1, 1, "orders", shape_id=1)

    db.put_record(1, 1, {"id": 1, "customer": "Alice", "amount": 19.9})      # dict by field name
    db.put(1, 2, [2, "Bob", 5.5], shape_id=1)                                  # list in field order
    db.put_dense_batch(1, 3, [[3, "Carol", 7.25], [4, "Dave", 120.0]], shape_id=1)  # u = 3, 4
    db.commit()

    print(db.get_record(1, 4))       # via layout 1, object 4
    print(db.get(1, 2))              # via realm 1, u = 2
    print(db.get_object(1, 3)["customer"])

{'_realm': 1, '_u': 4, '_lane': 0, '_lane_name': 'NL', '_shape': 1, 'id': 4, 'customer': 'Dave', 'amount': 120.0}
{'_realm': 1, '_u': 2, '_lane': 0, '_lane_name': 'NL', '_shape': 1, 'id': 2, 'customer': 'Bob', 'amount': 5.5}
Carol


*Expected output (from the cookbook):*

```text
{'_realm': 1, '_u': 4, '_lane': 0, '_lane_name': 'NL', '_shape': 1, 'id': 4, 'customer': 'Dave', 'amount': 120.0}
{'_realm': 1, '_u': 2, '_lane': 0, '_lane_name': 'NL', '_shape': 1, 'id': 2, 'customer': 'Bob', 'amount': 5.5}
Carol
```

`shape_id` can be omitted when the realm has a default shape:
`create_classic_layout` sets the layout's shape as default if the realm has
none yet, and `set_realm_default_shape(realm_id, shape_id)` changes it.

**Overwrite and delete.** Writing the same `u` again replaces the record
(the newest write wins). `delete_record(layout_id, object_id)` deletes a
record including its extension cells and returns the number of deleted
items (the dense record counts as one, plus one per extension cell);
`delete(realm_id, u)` hides the dense record at `u`.

In [9]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "people.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "people")
    db.create_shape(1, 1, "person", [
        {"v": 1, "name": "name", "type": "utf8"},
        {"v": 2, "name": "age", "type": "u8"},
    ])
    db.create_classic_layout(1, 1, "people", shape_id=1)
    db.put_dense_batch(1, 1, [["Ann", 31], ["Ben", 45], ["Cid", 27]], commit=True)

    db.put_record(1, 2, {"name": "Ben", "age": 46}, commit=True)   # birthday: replace u=2
    print(db.get_record(1, 2)["age"])

    print(db.delete_record(1, 1, commit=True))                      # number of deleted items
    print(db.get_record(1, 1))
    print(db.delete(1, 3, commit=True), db.get(1, 3))

46
1
None
True None


*Expected output (from the cookbook):*

```text
46
1
None
True None
```

**Enums.** A field can map labels to integer codes. You write and read
labels; the file stores the codes.

In [10]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "tickets.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "tickets")
    db.create_shape(1, 1, "ticket", [
        {"v": 1, "name": "title", "type": "utf8"},
        {"v": 2, "name": "state", "type": "u8", "enum": {"open": 0, "in_progress": 1, "done": 2}},
    ])
    db.create_classic_layout(1, 1, "tickets", shape_id=1)
    db.put_record(1, 1, {"title": "Fix login", "state": "in_progress"}, commit=True)
    print(db.get_record(1, 1)["state"])
    try:
        db.put_record(1, 2, {"title": "x", "state": "unknown"})
    except ValueError as e:
        print("ValueError:", e)

in_progress
ValueError: state: unknown enum value 'unknown'


*Expected output (from the cookbook):*

```text
in_progress
ValueError: state: unknown enum value 'unknown'
```

**Default shape, raw views and file positions.** A realm can have several
shapes; `set_realm_default_shape` selects the one used when `shape_id` is
omitted. `get_internal` shows the raw values by `v`; `committed_end`,
`last_valid_end` (end of the written data, including uncommitted blocks)
and `next_block_id` describe the container state.

In [11]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "internal.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "r")
    db.create_shape(1, 1, "small", [{"v": 1, "name": "x", "type": "u8"}])
    db.create_shape(2, 1, "large", [{"v": 1, "name": "x", "type": "u8"}, {"v": 2, "name": "y", "type": "u8"}])
    db.set_realm_default_shape(1, 2)
    db.create_classic_layout(1, 1, "c", shape_id=2)
    db.put(1, 1, [1, 2])                               # uses the default shape 2
    print(db.realms[1]["default_shape_id"], db.get_internal(1, 1)["by_v"])
    db.sync()                                          # written to the file, not committed
    print(db.last_valid_end > db.committed_end, db.next_block_id)

2 {1: 1, 2: 2}
True 4


*Expected output (from the cookbook):*

```text
2 {1: 1, 2: 2}
True 4
```

## 7. Single cells, lanes, null and delete

Any cell `(R, u, v, lane)` can be written directly, also outside a shape.
If the cell is not described by a shape field or an extension, pass its
type with `type=` (the type is then stored with the value).

In [12]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "cells.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "grid")
    db.put_cell(1, 14, 744, "left", type="utf8")                # lane NL (default)
    db.put_cell(1, 14, 744, "right", type="utf8", lane="NR")    # same (u, v), other lane
    db.put_cell(1, 14, 745, None, type="utf8")                  # explicit null
    db.commit()

    print(db.get_value(1, 14, 744), db.get_value(1, 14, 744, lane="NR"))
    print(db.get_value(1, 14, 746))                              # never written -> None
    cell = db.get_cell(1, 14, 745, dyn_address=False)
    print(cell["present"], cell["value"])                        # null: present, value None
    print(db.get_cell(1, 14, 746, dyn_address=False))            # missing -> None

    db.delete_cell(1, 14, 744, commit=True)                      # tombstone in lane NL only
    print(db.get_value(1, 14, 744), db.get_value(1, 14, 744, lane="NR"))

left right
None
True None
None
None right


*Expected output (from the cookbook):*

```text
left right
None
True None
None
None right
```

Lanes can be given as `"NL"`/`"NR"`, `0`/`1` (also lower case). Without
`type=` and without a declared field the write is rejected:

In [13]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "t.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "r")
    try:
        db.put_cell(1, 1, 1, 42)
    except nhydb.SchemaError as e:
        print("SchemaError:", e)

SchemaError: R1,u1,v1: cannot derive a data type; specify type=...


*Expected output (from the cookbook):*

```text
SchemaError: R1,u1,v1: cannot derive a data type; specify type=...
```

**Many cells at once.** `put_points(realm_id, cells)` writes a list of
tuples `(u, v, value)` or `(u, v, value, type)`. `delete_point` is the
realm-level counterpart of `delete_cell`.

In [14]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "points.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "sensors")
    report = db.put_points(1, [
        (1, 1, 20.5, "f64"),
        (1, 2, 21.0, "f64"),
        (2, 1, "offline", "utf8"),
        (10**30, 7, b"\x00\x01", "bytes"),       # coordinates may be arbitrarily large
    ], commit=True)
    print(report["cells"], report["u_groups"])
    print(db.get_value(1, 1, 2), db.get_value(1, 10**30, 7))
    db.delete_point(1, 2, 1, commit=True)
    print(db.get_value(1, 2, 1))

4 3
21.0 b'\x00\x01'
None


*Expected output (from the cookbook):*

```text
4 3
21.0 b'\x00\x01'
None
```

> **Note:** `get(realm, u)` and `get_record(...)` return **dense**
> records. A `u` that only has sparse cells (written with `put_cell`/
> `put_points`) returns `None` there; read such cells with `get_value` or
> `get_cell`, or iterate them with `iter_materialized_cells` (section 12).

## 8. Extensions: additional fields per record

Extensions are named, typed fields of a realm at a `v` above the BASE
fields. They are stored sparsely, so only records that have a value pay for
it. Reserve a `v` range for future BASE fields with `base_reserve_to`.

In [15]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "ext.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "products", base_reserve_to=20)
    db.create_shape(1, 1, "product", [
        {"v": 1, "name": "sku", "type": "utf8"},
        {"v": 2, "name": "price", "type": "f64"},
    ])
    db.create_classic_layout(1, 1, "products", shape_id=1)
    db.define_extension(1, 30, "utf8", name="color")
    db.define_extensions(1, [{"v": 31, "name": "weight_kg", "type": "f64"},
                             {"v": 32, "name": "tags", "type": "utf8"}])
    db.put_dense_batch(1, 1, [["A-100", 9.99], ["B-200", 24.5], ["C-300", 3.2]])

    db.put_extensions(1, 1, {"color": "red", "weight_kg": 0.4})       # by name
    db.put_extension_list(1, 2, [30, 31], ["blue", 1.25])             # by v
    db.put_extension_range(1, 3, 30, ["green", 0.1, "sale"])          # v = 30, 31, 32
    db.commit()

    print(db.get(1, 1))
    print(db.get(1, 3, include_extensions=False))
    print([e["name"] for e in db.extension_catalog(1)], db.resolve_v(1, "tags"))

    db.put_extensions(1, 1, {"tags": "new"}, merge=False, commit=True)  # replace all extensions
    print({k: v for k, v in db.get(1, 1).items() if not k.startswith("_")})

{'_realm': 1, '_u': 1, '_lane': 0, '_lane_name': 'NL', '_shape': 1, 'sku': 'A-100', 'price': 9.99, 'color': 'red', 'weight_kg': 0.4}
{'_realm': 1, '_u': 3, '_lane': 0, '_lane_name': 'NL', '_shape': 1, 'sku': 'C-300', 'price': 3.2}
['color', 'weight_kg', 'tags'] 32
{'sku': 'A-100', 'price': 9.99, 'tags': 'new'}


*Expected output (from the cookbook):*

```text
{'_realm': 1, '_u': 1, '_lane': 0, '_lane_name': 'NL', '_shape': 1, 'sku': 'A-100', 'price': 9.99, 'color': 'red', 'weight_kg': 0.4}
{'_realm': 1, '_u': 3, '_lane': 0, '_lane_name': 'NL', '_shape': 1, 'sku': 'C-300', 'price': 3.2}
['color', 'weight_kg', 'tags'] 32
{'sku': 'A-100', 'price': 9.99, 'tags': 'new'}
```

With `merge=True` (default) only the given extensions are written; with
`merge=False` the other live extensions of the record are deleted.
`put_record` and `put_dense_batch` (via `extensions_by_u`) can write
extensions together with the BASE fields:

In [16]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "ext2.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "r")
    db.create_shape(1, 1, "s", [{"v": 1, "name": "name", "type": "utf8"}])
    db.create_classic_layout(1, 1, "c", shape_id=1)
    db.define_extension(1, 10, "utf8", name="note")
    db.put_record(1, 1, {"name": "x", "note": "via put_record"})
    db.put(1, 2, ["y"], extensions={"note": "via put"})
    db.put_dense_batch(1, 3, [["z"]], extensions_by_u={3: {"note": "via batch"}})
    db.commit()
    print([db.get_value(1, u, 10) for u in (1, 2, 3)])

['via put_record', 'via put', 'via batch']


*Expected output (from the cookbook):*

```text
['via put_record', 'via put', 'via batch']
```